In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dataset/preprocessed_data.csv').drop(columns=['Unnamed: 0'])
categorical_cols = ['Car Name', 'Fuel', 'Location', 'Drive' , 'Type']
for col in categorical_cols:
    df[col] = df[col].astype('category')
    
X = df.iloc[:,:-1]
y = df.iloc[:,-1]


Code for regressionTree assuming all the columns are numerical and there are no categorical columns.

In [2]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
from joblib import Parallel, delayed

# Node class (unchanged)
class Node:
    def __init__(self, value=None, feature_index=None, threshold=None, left=None, right=None):
        self.value = value
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right

class RegressionTree:
    def __init__(self, max_depth=5, min_samples_split=5):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.max_features = None

    def fit(self, X, y):
        self.n_features = X.shape[1]
        self.root = self._split_node(X, y, 0)

    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return X.apply(self._predict_row, axis=1)

    def _split_node(self, X, y, depth):
        n_samples = X.shape[0]

        if depth >= self.max_depth or n_samples < self.min_samples_split:
            leaf_value = y.mean()
            return Node(value=leaf_value)
        
        best_feature_index, best_threshold = self._find_best_split(X, y)

        if best_feature_index is None:
            leaf_value = y.mean()
            return Node(value=leaf_value)
        
        feature = X.iloc[:, best_feature_index]
        if isinstance(best_threshold, set):
            left_idx = feature.isin(best_threshold)
            right_idx = ~left_idx
        else:
            left_idx = feature <= best_threshold
            right_idx = feature > best_threshold

        left_child = self._split_node(X[left_idx], y[left_idx], depth + 1)
        right_child = self._split_node(X[right_idx], y[right_idx], depth + 1)

        return Node(feature_index=best_feature_index, threshold=best_threshold, left=left_child, right=right_child)
    
    def _find_best_split(self, X, y):
        best_mse = np.inf
        best_feature_index = None
        best_threshold = None
        best_is_categorical = False

        feature_indices = np.arange(self.n_features)
        if self.max_features is not None:
            n_features_to_consider = min(self.max_features, self.n_features)
            feature_indices = np.random.choice(self.n_features, size=n_features_to_consider, replace=False)

        for feature_index in feature_indices:
            feature = X.iloc[:, feature_index]
            is_categorical = feature.dtype.name == 'category' or feature.dtype == 'object'
            
            if is_categorical:
                category_means = y.groupby(feature, observed=True).mean()  # Change to observed=False if needed
                sorted_categories = category_means.sort_values().index.tolist()

                for i in range(1, len(sorted_categories)):
                    left_categories = set(sorted_categories[:i])
                    left_idx = feature.isin(left_categories)
                    right_idx = ~left_idx

                    if left_idx.sum() == 0 or right_idx.sum() == 0:
                        continue

                    y_left = y[left_idx]
                    y_right = y[right_idx]
                    mse = (len(y_left) * y_left.var() + len(y_right) * y_right.var()) / len(y)
                    if mse < best_mse:
                        best_mse = mse 
                        best_feature_index = feature_index 
                        best_threshold = left_categories
                        best_is_categorical = True
            else:
                unique_values = np.unique(feature)

                for threshold in unique_values:
                    left_idx = feature <= threshold
                    right_idx = feature > threshold

                    if left_idx.sum() == 0 or right_idx.sum() == 0:
                        continue

                    y_left = y[left_idx]
                    y_right = y[right_idx]
                    mse = (len(y_left) * y_left.var() + len(y_right) * y_right.var()) / len(y)

                    if mse < best_mse:
                        best_mse = mse 
                        best_feature_index = feature_index 
                        best_threshold = threshold
                        best_is_categorical = False

        self.best_is_categorical = best_is_categorical
        return best_feature_index, best_threshold    
    
    def _predict_row(self, row):
        node = self.root 

        while node.value is None:
            val = row.iloc[node.feature_index]
            if isinstance(node.threshold, set):
                if val in node.threshold:
                    node = node.left 
                else:
                    node = node.right 
            else:
                if val <= node.threshold:
                    node = node.left 
                else:
                    node = node.right

        return node.value

class RandomForestRegressor:
    def __init__(self, n_estimators=100, max_depth=5, min_samples_split=5, max_features=None, n_jobs=-1):
        """
        Parameters:
        - n_estimators: number of trees in the forest
        - max_depth: maximum depth of each tree
        - min_samples_split: minimum number of samples required to split a node
        - max_features: number of features to consider for each split (integer or None for all features)
        - n_jobs: number of jobs to run in parallel (-1 means use all available cores)
        """
        if max_features is not None and (not isinstance(max_features, int) or max_features <= 0):
            raise ValueError("max_features must be a positive integer or None")
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.trees = []
        
    def _train_tree(self, X, y, max_features):
        """
        Helper function to train a single tree on a bootstrap sample.
        """
        # Debug input types
        if not isinstance(X, (pd.DataFrame, np.ndarray)) or not isinstance(y, (pd.Series, np.ndarray)):
            raise ValueError(f"Invalid input types: X={type(X)}, y={type(y)}")
        
        # Bootstrap sample
        X_sample, y_sample = resample(X, y, n_samples=X.shape[0])
        
        # Create and fit tree
        tree = RegressionTree(
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split
        )
        
        # Set max_features for the tree
        tree.max_features = max_features
        
        # Fit the tree
        tree.fit(X_sample, y_sample)
        return tree
    
    def fit(self, X, y):
        """
        Build a forest of trees from the training set (X, y) with parallel processing.
        """
        # Validate inputs
        if not isinstance(X, (pd.DataFrame, np.ndarray)):
            raise ValueError(f"X must be a pandas DataFrame or numpy array, got {type(X)}")
        if not isinstance(y, (pd.Series, np.ndarray)):
            raise ValueError(f"y must be a pandas Series or numpy array, got {type(y)}")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"X and y must have the same number of samples, got {X.shape[0]} and {y.shape[0]}")
        
        self.trees = []
        n_features = X.shape[1]
        
        # Determine the number of features to consider for each split
        if self.max_features is None:
            max_features = n_features
        else:
            max_features = min(self.max_features, n_features)
        
        # Train trees in parallel
        self.trees = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_tree)(X, y, max_features)
            for _ in range(self.n_estimators)
        )
    
    def predict(self, X):
        """
        Predict regression target for X.
        """
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(predictions, axis=0)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Assuming X, y are your data (pandas DataFrame and Series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize with max_features as an integer
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=8,
    min_samples_split=10,
    max_features=5,
    n_jobs=-1
)

# Train
rf.fit(X_train, y_train)

# Predict
predictions = rf.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, predictions)
print(f"Test MAE: {mae}")

Test MAE: 89373.5657440447


In [6]:
import joblib
joblib.dump(rf, 'random_forest_model.joblib')

['random_forest_model.joblib']